<div style="background: linear-gradient(135deg, #0D2137 0%, #1A4A7A 50%, #0D2137 100%); padding: 30px; border-radius: 12px; color: white; font-family: 'Segoe UI', sans-serif; border-left: 6px solid #F4C430;">
  <div style="font-size: 12px; color: #F4C430; letter-spacing: 3px; text-transform: uppercase; font-weight: 600;">Módulo Analítico y Pipeline ETL · Matías Retamal</div>
  <h1 style="margin: 10px 0 8px; font-size: 32px; font-weight: 700;">🔬 NHANES: Laboratory & Limited Access Data</h1>
  <p style="color: #caf0f8; font-size: 16px; margin: 0;">EDA End-to-End · Ciclos 2017-2018 y 2019-2020 · Feature Engineering para Longevidad</p>
</div>

---
## 📑 Índice
1. [Configuración del Entorno](#1)
2. [Ingesta de Datos Bronze](#2)
3. [Análisis de Calidad del Dato](#3)
4. [Distribuciones y Correlaciones](#4)
5. [Variables Clave para Longevidad](#5)
6. [Reflexiones Éticas y de Negocio](#6)
7. [Conclusiones y Próximos Pasos](#7)

In [ ]:
# ==============================================================================
# CELDA 1 — Configuración del Entorno
# Autor: Matías Retamal
# Objetivo: Importar librerías y definir la paleta corporativa del proyecto.
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# --- Paleta corporativa NHANES Health Analytics ---
CORP_PALETTE = {
    'primary':   '#1A4A7A',
    'accent':    '#F4C430',
    'dark':      '#0D2137',
    'light':     '#caf0f8',
    'danger':    '#E63946',
    'ok':        '#2DC653',
    'neutral':   '#6C757D',
}
SEQUENTIAL = ['#0D2137', '#1A4A7A', '#2874A6', '#5DADE2', '#A9CCE3', '#caf0f8']

# Estilo global matplotlib
plt.rcParams.update({
    'figure.facecolor':  CORP_PALETTE['dark'],
    'axes.facecolor':    '#0f2a40',
    'axes.edgecolor':    CORP_PALETTE['primary'],
    'axes.labelcolor':   CORP_PALETTE['light'],
    'xtick.color':       CORP_PALETTE['light'],
    'ytick.color':       CORP_PALETTE['light'],
    'text.color':        'white',
    'grid.color':        '#1e3a5a',
    'grid.linestyle':    '--',
    'grid.alpha':        0.5,
    'axes.titlesize':    14,
    'axes.titleweight':  'bold',
    'font.family':       'sans-serif',
})

RAW_DIR = '../data/01_raw'
print('✅ Entorno configurado correctamente.')
print(f'   → Directorio de datos crudos: {os.path.abspath(RAW_DIR)}')

<a id='2'></a>
## 🥉 2. Ingesta de Datos Bronze
> **[DATA-01 / DATA-04]** Lectura de archivos crudos de Laboratorio y Limited Access para ambos ciclos. Los datos son inmutables en esta etapa.

In [ ]:
# ==============================================================================
# CELDA 2 — Ingesta Bronze
# Archivos de Laboratorio (LBXXX) y Limited Access (SSXX / sintéticos)
# NHANES suffixes: _J = 2017-2018 | _K = 2019-2020
# ==============================================================================

LAB_FILES = {
    '2017_2018': ['cbc_2017_2018', 'biopro_2017_2018', 'trigly_2017_2018',
                  'ghb_2017_2018', 'hdl_2017_2018'],
    '2019_2020': ['cbc_2019_2020', 'biopro_2019_2020', 'trigly_2019_2020',
                  'ghb_2019_2020', 'hdl_2019_2020'],
}

LIMITED_FILES = {
    '2017_2018': ['mort_2017_2018_public'],
    '2019_2020': ['mort_2019_2020_public'],
}

def load_parquet_safe(filepath: str) -> pd.DataFrame:
    """Carga un Parquet de forma segura; retorna DataFrame vacío si no existe."""
    if os.path.exists(filepath):
        return pd.read_parquet(filepath)
    print(f'  ⚠️  Archivo no encontrado: {filepath} — se omite.')
    return pd.DataFrame()

bronze_lab  = {}
bronze_ltd  = {}

for cycle, files in LAB_FILES.items():
    frames = [load_parquet_safe(os.path.join(RAW_DIR, f'{f}.parquet')) for f in files]
    frames = [df for df in frames if not df.empty]
    if frames:
        base = frames[0].copy()
        for extra in frames[1:]:
            base = base.merge(extra[['SEQN'] + [c for c in extra.columns if c != 'SEQN']],
                              on='SEQN', how='outer')
        base['cycle_year'] = cycle.replace('_', '-')
        bronze_lab[cycle] = base
        print(f'✅ LAB  {cycle}: {base.shape[0]:,} filas · {base.shape[1]} cols')
    else:
        print(f'⚠️  LAB  {cycle}: sin datos en Bronze.')

for cycle, files in LIMITED_FILES.items():
    frames = [load_parquet_safe(os.path.join(RAW_DIR, f'{f}.parquet')) for f in files]
    frames = [df for df in frames if not df.empty]
    if frames:
        df_ltd = pd.concat(frames, ignore_index=True)
        df_ltd['cycle_year'] = cycle.replace('_', '-')
        bronze_ltd[cycle] = df_ltd
        print(f'✅ LTD  {cycle}: {df_ltd.shape[0]:,} filas · {df_ltd.shape[1]} cols')
    else:
        print(f'⚠️  LTD  {cycle}: sin datos en Bronze.')

<a id='3'></a>
## 🔍 3. Análisis de Calidad del Dato
> Auditamos nulos, rangos esperados y encodings SAS antes de tocar los datos.

In [ ]:
# ==============================================================================
# CELDA 3 — Quality Audit: Nulos, Rangos y Outliers
# ==============================================================================

def audit_dataframe(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """
    Genera un reporte de calidad para un DataFrame.

    Args:
        df:   DataFrame a auditar.
        name: Nombre descriptivo del dataset.

    Returns:
        pd.DataFrame: Reporte con métricas de calidad por columna.
    """
    report = pd.DataFrame({
        'columna':         df.columns,
        'dtype':           df.dtypes.values,
        'nulos_abs':       df.isna().sum().values,
        'nulos_pct':       (df.isna().mean() * 100).round(2).values,
        'unicos':          df.nunique().values,
        'min':             df.select_dtypes(include='number').min().reindex(df.columns).values,
        'max':             df.select_dtypes(include='number').max().reindex(df.columns).values,
    })
    print(f'\n📊 Reporte de Calidad — {name}')
    print(f'   Shape: {df.shape} | Duplicados SEQN: {df["SEQN"].duplicated().sum() if "SEQN" in df.columns else "N/A"}')
    return report

audit_reports = {}
for cycle, df in bronze_lab.items():
    audit_reports[f'lab_{cycle}'] = audit_dataframe(df, f'Laboratorio {cycle}')
    display(audit_reports[f'lab_{cycle}'].sort_values('nulos_pct', ascending=False).head(15))

for cycle, df in bronze_ltd.items():
    audit_reports[f'ltd_{cycle}'] = audit_dataframe(df, f'Limited Access {cycle}')

In [ ]:
# ==============================================================================
# CELDA 4 — Visualización: Heatmap de Nulos por Dataset
# ==============================================================================

def plot_null_heatmap(df: pd.DataFrame, title: str, max_cols: int = 30):
    """Heatmap de valores nulos para auditoría visual."""
    cols_with_nulls = df.columns[df.isna().any()].tolist()[:max_cols]
    if not cols_with_nulls:
        print(f'  ✅ {title}: sin columnas con nulos.')
        return

    fig, ax = plt.subplots(figsize=(16, 6))
    null_matrix = df[cols_with_nulls].isna().astype(int)
    sns.heatmap(
        null_matrix.T,
        cmap=['#1A4A7A', '#F4C430'],
        cbar_kws={'label': '0=Presente  1=Nulo'},
        linewidths=0,
        ax=ax,
        yticklabels=True,
    )
    ax.set_title(f'🔍 Patrón de Nulos — {title}', pad=15, color=CORP_PALETTE['accent'])
    ax.set_xlabel('Observaciones (muestra)', labelpad=10)
    plt.tight_layout()
    plt.show()

for cycle, df in bronze_lab.items():
    plot_null_heatmap(df.sample(min(500, len(df)), random_state=42), f'Laboratorio {cycle}')

<a id='4'></a>
## 📈 4. Distribuciones y Correlaciones

In [ ]:
# ==============================================================================
# CELDA 5 — Distribuciones de Variables de Laboratorio Clave
# Variables de interés para longevidad: HbA1c, Colesterol, Creatinina, etc.
# ==============================================================================

# Mapeo de variables NHANES a nombre legible
LAB_VARS_LONGEVITY = {
    'LBXGH':  'HbA1c (%) — Glucemia crónica',
    'LBXTC':  'Colesterol Total (mg/dL)',
    'LBDHDL': 'HDL Colesterol (mg/dL)',
    'LBXTR':  'Triglicéridos (mg/dL)',
    'LBXSCR': 'Creatinina Sérica (mg/dL)',
    'LBXSGL': 'Glucosa Sérica (mg/dL)',
    'LBXWBCSI':'Glóbulos Blancos (1000 cél/µL)',
    'LBXHGB': 'Hemoglobina (g/dL)',
}

def plot_distributions(df: pd.DataFrame, var_map: dict, title_suffix: str):
    """Histogramas con KDE para variables numéricas clave."""
    available = {k: v for k, v in var_map.items() if k in df.columns}
    if not available:
        print('  ⚠️  Ninguna variable de longevidad encontrada en este ciclo.')
        return

    n = len(available)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
    axes = axes.flatten()

    for idx, (var, label) in enumerate(available.items()):
        data = df[var].dropna()
        ax = axes[idx]
        ax.hist(data, bins=50, color=CORP_PALETTE['primary'], alpha=0.8, density=True, edgecolor='none')
        data.plot.kde(ax=ax, color=CORP_PALETTE['accent'], linewidth=2)
        ax.set_title(label, fontsize=10, color=CORP_PALETTE['accent'])
        ax.set_xlabel(var, fontsize=8)
        ax.axvline(data.mean(), color=CORP_PALETTE['ok'], linestyle='--', linewidth=1.5, label=f'Media: {data.mean():.2f}')
        ax.axvline(data.median(), color=CORP_PALETTE['danger'], linestyle=':', linewidth=1.5, label=f'Mediana: {data.median():.2f}')
        ax.legend(fontsize=7, loc='upper right')

    for j in range(idx+1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f'📊 Distribuciones de Variables para Longevidad — {title_suffix}',
                 y=1.02, fontsize=15, color=CORP_PALETTE['accent'], fontweight='bold')
    plt.tight_layout()
    plt.show()

for cycle, df in bronze_lab.items():
    plot_distributions(df, LAB_VARS_LONGEVITY, f'Laboratorio {cycle}')

In [ ]:
# ==============================================================================
# CELDA 6 — Matriz de Correlación de Variables de Longevidad
# ==============================================================================

def plot_correlation_matrix(df: pd.DataFrame, var_map: dict, title: str):
    """Heatmap de correlación de Pearson para variables numéricas seleccionadas."""
    available_cols = [c for c in var_map.keys() if c in df.columns]
    if len(available_cols) < 2:
        print('  ⚠️  Insuficientes variables para correlación.')
        return

    corr = df[available_cols].corr()
    labels = [var_map.get(c, c) for c in available_cols]

    fig, ax = plt.subplots(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(
        corr,
        mask=mask,
        cmap='coolwarm',
        center=0,
        annot=True,
        fmt='.2f',
        linewidths=0.5,
        ax=ax,
        xticklabels=[v.split('—')[0].strip() for v in labels],
        yticklabels=[v.split('—')[0].strip() for v in labels],
        annot_kws={'size': 9},
        cbar_kws={'shrink': 0.8},
    )
    ax.set_title(f'🔗 Correlaciones — {title}', pad=15, color=CORP_PALETTE['accent'])
    plt.tight_layout()
    plt.show()

for cycle, df in bronze_lab.items():
    plot_correlation_matrix(df, LAB_VARS_LONGEVITY, f'Laboratorio {cycle}')

<a id='5'></a>
## 🎯 5. Variables Clave para Longevidad

> **Reflexión analítica:** Las variables de laboratorio son biomarcadores objetivos y auditables. A diferencia de datos auto-reportados, no tienen sesgo de respuesta. Las seleccionadas responden a evidencia clínica:
> - **HbA1c > 6.5%** → diagnóstico de diabetes, fuertemente asociada a mortalidad prematura.
> - **HDL < 40 mg/dL (hombres) / <50 (mujeres)** → factor de riesgo cardiovascular.
> - **Creatinina sérica > 1.2 mg/dL** → marcador de daño renal crónico.
> - **Triglicéridos > 200 mg/dL** → síndrome metabólico.

In [ ]:
# ==============================================================================
# CELDA 7 — Análisis de Percentiles y Umbrales Clínicos
# ==============================================================================

CLINICAL_THRESHOLDS = {
    'LBXGH':  {'alto': 6.5,  'muy_alto': 8.0,  'label': 'HbA1c'},
    'LBDHDL': {'bajo': 40.0, 'muy_bajo': 30.0, 'label': 'HDL'},
    'LBXTR':  {'alto': 150.0,'muy_alto': 200.0, 'label': 'Triglicéridos'},
    'LBXSCR': {'alto': 1.2,  'muy_alto': 2.0,  'label': 'Creatinina'},
}

all_lab = pd.concat(bronze_lab.values(), ignore_index=True) if bronze_lab else pd.DataFrame()

if not all_lab.empty:
    summary_rows = []
    for var, rules in CLINICAL_THRESHOLDS.items():
        if var not in all_lab.columns:
            continue
        col = all_lab[var].dropna()
        row = {'Variable': rules['label'], 'NHANES_Code': var, 'N': len(col),
               'Media': col.mean(), 'Mediana': col.median(), 'Std': col.std(),
               'P5': col.quantile(0.05), 'P95': col.quantile(0.95)}
        if 'alto' in rules:
            row['% en riesgo'] = f"{(col > rules['alto']).mean()*100:.1f}%"
        else:
            row['% en riesgo'] = f"{(col < rules['bajo']).mean()*100:.1f}%"
        summary_rows.append(row)

    if summary_rows:
        display(pd.DataFrame(summary_rows).set_index('Variable').round(3))
else:
    print('  ⚠️  No hay datos combinados disponibles todavía.')

<a id='6'></a>
## ⚖️ 6. Reflexiones Éticas y de Negocio

| Decisión | Justificación Ética | Impacto en Pipeline |
|---|---|---|
| Eliminar variables de raza en features predictivos | Evitar discriminación algorítmica (proxy variable) | Se excluyen `RIDRETH1`/`RIDRETH3` de Gold |
| Tratar códigos SAS 7/9/77/99 como `NaN` | Son "No sabe/No responde", no datos reales | Imputación estadística en Silver |
| Separar registros rechazados (no eliminar) | Auditabilidad y reproducibilidad del pipeline | `member3_rejected.parquet` |
| Limitar features a biomarcadores objetivos | Reducir sesgo de auto-reporte | Solo variables de laboratorio en Gold |
| No imputar mortalidad sintética | Evitar data leakage en variable objetivo | `mort_*` solo en inferencia final |

<a id='7'></a>
## ✅ 7. Conclusiones y Próximos Pasos

- **Bronze completada:** datos crudos intactos y documentados para 2017-2018 y 2019-2020.
- **Variables críticas identificadas:** HbA1c, HDL, Triglicéridos y Creatinina son los mejores predictores de longevidad disponibles en NHANES Laboratory.
- **Calidad del dato:** Se detectaron patrones de nulidad sistemáticos en variables de acceso limitado → requieren imputación en Silver.
- **Siguiente fase:** Ejecutar `kedro run --pipeline=processing_m3` para procesar Bronze → Silver → Gold con los nodos de validación y feature engineering.

---
<div style="background: #0D2137; border-left: 4px solid #F4C430; padding: 12px; border-radius: 6px; color: #caf0f8; font-size: 13px;">
  📌 <strong>Autor:</strong> Matías Retamal · NHANES Health Analytics · Rama: <code>la-cabra-🔥🐐</code>
</div>